# Lab 6: Actor-Critic on LunarLanderContinuous-v2

## Design Constraints

- A trained Agent must be saveable and restorable from disk (weights + config) so visualizations can be reproduced in the notebook without retraining.
- All hyperparameter choices (learning rates, architecture, activation functions, initial log-std, entropy regularization) must be documented in the notebook.

In [9]:
import warnings
warnings.filterwarnings("ignore")

In [10]:
# Project Structure
# DONE: Establish /src, /test, /data, /img directory layout

! tree ..

..
├── data
├── img
├── ipynb
│   ├── development.ipynb
│   └── report.ipynb
├── README.md
├── src
│   ├── __init__.py
│   ├── __pycache__
│   │   ├── __init__.cpython-311.pyc
│   │   └── agent.cpython-311.pyc
│   └── agent.py
└── test
    ├── __pycache__
    │   ├── conftest.cpython-311-pytest-9.0.2.pyc
    │   ├── test_actor.cpython-311-pytest-9.0.2.pyc
    │   └── test_critic.cpython-311-pytest-9.0.2.pyc
    ├── conftest.py
    ├── test_actor.py
    └── test_critic.py

8 directories, 13 files


In [3]:
# DONE: Verify PyTorch and Gymnasium are available; instantiate LunarLanderContinuous-v2
#       and inspect observation space, action space shape, and action bounds
import torch
import gymnasium as gym

print(f"PyTorch  : {torch.__version__}")
print(f"Gymnasium: {gym.__version__}")

env = gym.make("LunarLanderContinuous-v3")
obs_space = env.observation_space
act_space = env.action_space

print(f"\nObservation space : {obs_space}")
print(f"Action space      : {act_space}")
print(f"Action shape      : {act_space.shape}")
print(f"Action low        : {act_space.low}")
print(f"Action high       : {act_space.high}")
env.close()

PyTorch  : 2.10.0
Gymnasium: 1.2.3

Observation space : Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Action space      : Box(-1.0, 1.0, (2,), float32)
Action shape      : (2,)
Action low        : [-1. -1.]
Action high       : [1. 1.]


### Actor Hyperparameters

| Parameter | Value | Notes |
|---|---|---|
| `obs_dim` | 8 | LunarLanderContinuous-v3 observation space |
| `act_dim` | 2 | Main engine throttle, side engine throttle |
| `hidden_sizes` | `(64, 64)` | Two hidden layers; starting point for empirical tuning |
| `activation` | `ReLU` | Fast training; Tanh vs ReLU is a candidate experiment |
| `state_dependent_std` | `False` | Free-parameter mode; single global log-std shared across all states |
| Initial `log_std` | `0.0` | `exp(0) = 1.0`; broad initial exploration |
| `log_std` clamp | `[-20, 2]` | Prevents numerical instability from extreme variance estimates |

In [4]:
# Actor
# DONE: Actor produces a Gaussian policy (mean and log-std) over continuous actions
# DONE: Tests for Actor
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import gymnasium as gym
from torch.distributions import Normal
from src.agent import Actor

# --- environment bounds ---
env = gym.make("LunarLanderContinuous-v3")
obs_dim  = env.observation_space.shape[0]
act_dim  = env.action_space.shape[0]
act_low  = torch.tensor(env.action_space.low)
act_high = torch.tensor(env.action_space.high)
env.close()

torch.manual_seed(42)
actor_head = Actor(obs_dim, act_dim, state_dependent_std=True)
actor_free = Actor(obs_dim, act_dim, state_dependent_std=False)

obs1 = torch.zeros(obs_dim)
obs2 = torch.ones(obs_dim)

# --- forward pass ---
mu, log_std = actor_head(obs1)
print("=== Actor (state-dependent std) ===")
print(f"mu shape     : {mu.shape}   values: {mu.detach().numpy().round(3)}")
print(f"log_std shape: {log_std.shape}   values: {log_std.detach().numpy().round(3)}")

# --- sample, clip, log_prob, entropy ---
dist   = Normal(mu, log_std.exp())
action = dist.sample()
action_clipped = torch.clamp(action, act_low, act_high)
log_prob = dist.log_prob(action).sum(dim=-1)
entropy  = dist.entropy().sum(dim=-1)

print(f"\nRaw sample   : {action.detach().numpy().round(3)}")
print(f"Clipped      : {action_clipped.detach().numpy().round(3)}  (bounds [{act_low.numpy()}, {act_high.numpy()}])")
print(f"log_prob     : {log_prob.item():.4f}")
print(f"entropy      : {entropy.item():.4f}")

# --- mode contrast: does log_std vary with state? ---
print("\n=== Mode contrast ===")
_, log_std_head_obs1 = actor_head(obs1)
_, log_std_head_obs2 = actor_head(obs2)
_, log_std_free_obs1 = actor_free(obs1)
_, log_std_free_obs2 = actor_free(obs2)

print(f"Head  log_std(obs1): {log_std_head_obs1.detach().numpy().round(4)}")
print(f"Head  log_std(obs2): {log_std_head_obs2.detach().numpy().round(4)}  <- varies with state")
print(f"Free  log_std(obs1): {log_std_free_obs1.detach().numpy().round(4)}")
print(f"Free  log_std(obs2): {log_std_free_obs2.detach().numpy().round(4)}  <- same regardless of state")

=== Actor (state-dependent std) ===
mu shape     : torch.Size([2])   values: [ 0.08  -0.089]
log_std shape: torch.Size([2])   values: [0.002 0.09 ]

Raw sample   : [-0.215 -0.867]
Clipped      : [-0.215 -0.867]  (bounds [[-1. -1.], [1. 1.]])
log_prob     : -2.2255
entropy      : 2.9299

=== Mode contrast ===
Head  log_std(obs1): [0.0023 0.0898]
Head  log_std(obs2): [0.2263 0.154 ]  <- varies with state
Free  log_std(obs1): [0. 0.]
Free  log_std(obs2): [0. 0.]  <- same regardless of state


### Critic Hyperparameters

| Parameter | Value | Notes |
|---|---|---|
| `obs_dim` | 8 | LunarLanderContinuous-v3 observation space |
| `hidden_sizes` | `(64, 64)` | Two hidden layers; starting point for empirical tuning |
| `activation` | `ReLU` | Matches Actor; Tanh vs ReLU is a candidate experiment |
| Output | scalar V(s) | Single value estimate per observation |

In [ ]:
# Critic
# DONE: Critic estimates state value V(s)
# DONE: Tests for Critic

In [6]:
# Agent
# TODO: Agent composes Actor and Critic; supports action selection and per-step TD updates
# TODO: Agent clips sampled actions to environment bounds before interacting with the environment
# TODO: Agent can be saved and restored from disk (weights + config) to avoid retraining
# TODO: Tests for Agent

In [7]:
# Trainer
# TODO: Trainer runs the Actor-Critic loop on LunarLanderContinuous-v2
# TODO: Trainer collects step-level metrics (TD error, entropy) and episode-level metrics (return, trajectory)
# TODO: Each run is namespaced by label + timestamp; all config values appear as columns in the CSV for pandas comparison
# TODO: Tests for Trainer

In [8]:
# Visualizations (each exported as PNG to /img)
# TODO: Return per episode
# TODO: Policy entropy over training
# TODO: TD error magnitude over training
# TODO: Sample trajectories from the learned policy
# TODO: Actor mean action across the state space